# 05 · Chapter 3: The whole support office

Every message goes to the **cheapest helper that can answer it**:

```
message → Receptionist (ML model) ──at least 80% sure + simple──→ ready-made answer      ($0, ~0.05 ms)
              │
              └─40-80% sure, policy question──→ Junior assistant (local LLM)          ($0)
              │
              └─action / complaint / unsure──→ Senior expert (cloud LLM + tools)       (paid)
                                                      │
                                                      └─big refund / very upset──→ Manager (a human)
```

**My plan:**
1. See who would answer each message (free, no LLM)
2. Try one example per helper
3. Compare the expert alone (chapter 1) with the whole office on a mixed inbox

In [ ]:
import sys
sys.path.append("..")   # so Python can find the budget_agent folder

import pandas as pd
pd.set_option("display.max_colwidth", 80)
from dotenv import load_dotenv
load_dotenv("../.env")

from budget_agent import config, data
from budget_agent.router import Router

office = Router()
print("Junior assistant (Ollama) ready:", office.local_ready())

## Step 1: Who would answer? (no LLM is called here)

In [ ]:
for helper, message in config.DEMO_MESSAGES.items():
    layer, intent, confidence = office.choose_layer(message)
    print(f"{config.HELPERS[layer]:32} {intent:24} {confidence:4.0%}   {message}")

The manager's message also goes to the senior expert first. The expert then hands it to the manager, because the guardrail in `tools.py` says refunds above ₹10,000 need a human.

If Ollama isn't running, the junior's message goes to the senior expert instead.

### The routing rule, written by hand

The real rule is `Router.choose_layer()` in `budget_agent/router.py`. Here it is in a few lines, so I can see every decision:

In [ ]:
from budget_agent import classifier
from budget_agent.router import ORDER_INTENTS   # track_order, track_refund: they need an order lookup

def who_answers(text, junior_ready):
    intent, sure = classifier.predict(office.model, text)
    simple = intent not in config.ACTION_INTENTS                 # refunds, cancels... are never simple
    if simple and sure >= config.ML_CONFIDENCE:                  # 0.80
        return "ml"                                              # receptionist
    if simple and intent not in ORDER_INTENTS and sure >= config.LOCAL_LLM_CONFIDENCE and junior_ready:  # 0.40
        return "local_llm"                                       # junior: policy questions only
    return "cloud_llm"                                           # senior expert

test = data.load_messages().query("split == 'test'")
agrees = all(who_answers(t, office.local_ready()) == office.choose_layer(t)[0] for t in test.text)
print("My rule gives the same answer as router.py on all", len(test), "test messages:", agrees)

In [ ]:
helpers = [config.HELPERS[office.choose_layer(text)[0]] for text in test.text]
pd.Series(helpers).value_counts(normalize=True)

On normal messages, a big share never needs the expert at all.
(Refunds, cancels, complaints etc. always go to the expert, on purpose, because they change real things.)

## Step 2: One example per helper

In [ ]:
def show(r):
    who = config.HELPERS[r.layer]
    if r.escalated:                                   # the expert handed it to a human
        who += " → " + config.HELPERS["human"]
    print(f"[{who}] intent={r.intent} ({r.confidence:.0%})  {r.seconds:.3f}s  ${r.cost_usd:.4f}")
    print("  ", r.reply, "\n")

show(office.handle(config.DEMO_MESSAGES["ml"]))          # receptionist

In [ ]:
show(office.handle(config.DEMO_MESSAGES["local_llm"]))   # junior assistant (if Ollama is running)

In [ ]:
show(office.handle(config.DEMO_MESSAGES["cloud_llm"]))   # senior expert

In [ ]:
show(office.handle(config.DEMO_MESSAGES["human"]))       # senior expert → manager

## Step 3: The expert alone vs the whole office, on a mixed inbox

This calls the cloud LLM many times, so I start small (`--n 30` costs well under $1).
I run it from the terminal so it saves a CSV the app can show:

```
uv run python run.py benchmark --n 30
```

In [ ]:
results = pd.read_csv(config.DATA_DIR / "benchmark_results.csv")
summary = pd.DataFrame({
    "total cost ($)": [results.v2_cost.sum(), results.v4_cost.sum()],
    "average seconds": [results.v2_seconds.mean(), results.v4_seconds.mean()],
}, index=["Expert only (chapter 1)", "Whole office (chapter 3)"])
summary.round(3)

In [ ]:
print(f"The whole office is {results.v2_cost.sum() / results.v4_cost.sum():.1f}x cheaper")
results.v4_layer.map(config.HELPERS).value_counts()

In [ ]:
# Read some replies side by side, to check quality didn't drop
results[["message", "v4_layer", "v2_reply", "v4_reply"]].sample(5, random_state=1)

## What I learned
- Most of a support inbox is simple. The receptionist (a tiny ML model) handles it in milliseconds, for free.
- The office only pays for the senior expert when it's really needed: actions, complaints, unclear messages.
- The 80% rule is a **dial**: higher = safer but more expensive, lower = cheaper but riskier.
- This is **inference engineering**: the right model for each request, not one giant model for everything.